# LLM Training Recipes — Building Blocks of Modern Language Models

**Module 22** of the PyTorch Complete Learning Guide

This notebook walks through the key techniques used in modern LLMs (Llama, Mistral, GPT-4, etc.):
- RoPE (Rotary Position Embeddings)
- KV Cache for fast generation
- Grouped-Query Attention (GQA)
- SwiGLU FFN
- RMSNorm
- Weight Tying
- BFloat16 Training
- Gradient Accumulation
- Putting it all together: a Mini-LLM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

---
## 1. RoPE (Rotary Position Embeddings)

### The Idea

RoPE encodes position by **rotating** query and key vectors. The rotation angle depends on both the position and the dimension index:

$$\theta_i = 10000^{-2i/d}$$

For position $m$, dimension pair $i$: rotate by angle $m \cdot \theta_i$

The key insight: the dot product $q_m \cdot k_n$ after rotation depends only on the **relative** position $(m - n)$.

In [ ]:
def precompute_freqs_cis(dim: int, max_seq_len: int, theta: float = 10000.0):
    """Precompute complex exponentials for RoPE."""
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len)
    freqs = torch.outer(t, freqs)  # (seq_len, dim/2)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # e^(i*freq)
    return freqs_cis


def apply_rotary_emb(xq, xk, freqs_cis):
    """Apply rotary embeddings to Q and K."""
    xq_c = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_c = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    
    # Reshape freqs for broadcasting: (seq, dim/2) -> (1, seq, 1, dim/2)
    ndim = xq_c.ndim
    shape = [1] * ndim
    shape[1] = freqs_cis.shape[0]
    shape[-1] = freqs_cis.shape[1]
    freqs = freqs_cis.view(*shape)
    
    xq_out = torch.view_as_real(xq_c * freqs).flatten(-2)
    xk_out = torch.view_as_real(xk_c * freqs).flatten(-2)
    return xq_out.type_as(xq), xk_out.type_as(xk)


# Test it
head_dim = 64
seq_len = 16
freqs_cis = precompute_freqs_cis(head_dim, seq_len)
print(f"Freqs shape: {freqs_cis.shape}  (seq_len, head_dim/2)")
print(f"Each entry is a complex number: e^(i * position * theta_i)")

### Visualize Position Encoding

Let's see how RoPE rotation angles vary across positions and dimensions.

In [ ]:
# Visualize the rotation angles
head_dim = 64
max_len = 128
theta = 10000.0

freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
positions = torch.arange(max_len)
angles = torch.outer(positions, freqs)  # (seq, dim/2)

print("Rotation angles (radians) for first 8 positions, selected dimension pairs:")
print(f"{'Pos':<5}", end="")
for d in [0, 4, 8, 16, 24, 31]:
    print(f"{'dim '+str(d):<10}", end="")
print()
for p in range(8):
    print(f"{p:<5}", end="")
    for d in [0, 4, 8, 16, 24, 31]:
        print(f"{angles[p, d].item():<10.4f}", end="")
    print()

print("\nKey observation: low dimensions rotate fast, high dimensions rotate slowly")
print(f"Dim 0 wavelength: {2*math.pi/freqs[0]:.1f} tokens")
print(f"Dim 31 wavelength: {2*math.pi/freqs[31]:.0f} tokens")

### Verify Relative Position Property

In [ ]:
# Verify: attention score depends only on relative position
torch.manual_seed(42)
head_dim = 32
freqs_cis = precompute_freqs_cis(head_dim, 100)

q = torch.randn(1, 1, 1, head_dim)  # Single query vector
k = torch.randn(1, 1, 1, head_dim)  # Single key vector

# Place same vectors at different absolute positions, same relative distance
print("Verifying relative position invariance:")
print(f"{'(q_pos, k_pos)':<18} {'Relative dist':<15} {'Attention score':<18}")
print("-" * 50)

for q_pos, k_pos in [(5, 8), (15, 18), (50, 53), (90, 93)]:
    q_rot, _ = apply_rotary_emb(q, q, freqs_cis[q_pos:q_pos+1])
    _, k_rot = apply_rotary_emb(k, k, freqs_cis[k_pos:k_pos+1])
    score = (q_rot * k_rot).sum().item()
    print(f"({q_pos}, {k_pos}){'':10} {k_pos - q_pos:<15} {score:<18.6f}")

print("\nAll scores with same relative distance (3) are identical!")

---
## 2. KV Cache

### The Problem
During autoregressive generation, we generate one token at a time. Each new token needs to attend to ALL previous tokens. Without caching, we'd recompute K and V for all past tokens at every step — O(n²) total work.

### The Solution
Cache K and V after computing them. Each new step:
1. Compute Q, K, V for just the new token
2. Append K, V to cache
3. Attend: new Q against all cached K, V

In [ ]:
class KVCache:
    """Pre-allocated KV cache for autoregressive generation."""
    
    def __init__(self, batch_size, max_seq_len, n_kv_heads, head_dim, dtype=torch.float32):
        shape = (batch_size, max_seq_len, n_kv_heads, head_dim)
        self.k_cache = torch.zeros(shape, dtype=dtype)
        self.v_cache = torch.zeros(shape, dtype=dtype)
        self.pos = 0
    
    def update(self, k, v):
        """Append new K/V and return full cached tensors."""
        new_len = k.shape[1]
        self.k_cache[:, self.pos:self.pos + new_len] = k
        self.v_cache[:, self.pos:self.pos + new_len] = v
        self.pos += new_len
        return self.k_cache[:, :self.pos], self.v_cache[:, :self.pos]
    
    def reset(self):
        self.pos = 0


# Demonstrate prefill + decode
cache = KVCache(batch_size=1, max_seq_len=32, n_kv_heads=4, head_dim=16)
print(f"Cache allocated: {cache.k_cache.shape}")
print(f"Memory: {cache.k_cache.nelement() * 4 * 2 / 1024:.1f} KB (K+V)")

In [ ]:
# Step-by-step demonstration
print("=== Prefill Phase ===")
prompt_k = torch.randn(1, 8, 4, 16)  # 8 prompt tokens
prompt_v = torch.randn(1, 8, 4, 16)
cached_k, cached_v = cache.update(prompt_k, prompt_v)
print(f"After prefill: cache contains {cache.pos} tokens")
print(f"Cached K shape: {cached_k.shape}")

print("\n=== Decode Phase ===")
for step in range(4):
    new_k = torch.randn(1, 1, 4, 16)  # 1 new token
    new_v = torch.randn(1, 1, 4, 16)
    cached_k, cached_v = cache.update(new_k, new_v)
    print(f"Step {step+1}: cache has {cache.pos} tokens, K shape: {cached_k.shape}")

print(f"\nFinal: {cache.pos} tokens in cache (8 prefill + 4 generated)")

### Benchmark: Generation Speed With vs Without Cache

In [ ]:
class SimpleAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.wq = nn.Linear(dim, dim, bias=False)
        self.wk = nn.Linear(dim, dim, bias=False)
        self.wv = nn.Linear(dim, dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)
    
    def forward(self, x, cache=None):
        b, s, _ = x.shape
        q = self.wq(x).view(b, s, self.n_heads, self.head_dim)
        k = self.wk(x).view(b, s, self.n_heads, self.head_dim)
        v = self.wv(x).view(b, s, self.n_heads, self.head_dim)
        
        if cache is not None:
            k, v = cache.update(k, v)
        
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        scores = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1,2).reshape(b, s, -1)
        return self.wo(out)


dim, n_heads, seq_len = 256, 8, 64
attn = SimpleAttention(dim, n_heads)
attn.eval()
x = torch.randn(1, seq_len, dim)

# Without cache: recompute everything each step
start = time.perf_counter()
with torch.no_grad():
    for t in range(1, seq_len + 1):
        _ = attn(x[:, :t])
no_cache_time = time.perf_counter() - start

# With cache: only new token each step
cache = KVCache(1, seq_len, n_heads, dim // n_heads)
start = time.perf_counter()
with torch.no_grad():
    for t in range(seq_len):
        _ = attn(x[:, t:t+1], cache=cache)
cache_time = time.perf_counter() - start

print(f"Without cache: {no_cache_time*1000:.1f} ms")
print(f"With cache:    {cache_time*1000:.1f} ms")
print(f"Speedup:       {no_cache_time/cache_time:.1f}x")

---
## 3. Grouped-Query Attention (GQA)

Instead of having the same number of K/V heads as Q heads, GQA uses fewer K/V heads.
Multiple Q heads share the same K/V head.

- **MHA**: 32 Q heads, 32 KV heads (standard)
- **GQA**: 32 Q heads, 8 KV heads (Llama 2 70B, Llama 3)
- **MQA**: 32 Q heads, 1 KV head (extreme)

Benefits: smaller KV cache, less memory bandwidth during decode.

In [ ]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """Expand KV heads to match Q heads.
    x: (batch, seq, n_kv_heads, head_dim) -> (batch, seq, n_kv_heads * n_rep, head_dim)
    """
    if n_rep == 1:
        return x
    b, s, h, d = x.shape
    return x.unsqueeze(3).expand(b, s, h, n_rep, d).reshape(b, s, h * n_rep, d)


# Demo: 8 KV heads -> 32 Q heads (repeat 4x)
kv = torch.randn(1, 16, 8, 64)  # 8 KV heads
kv_expanded = repeat_kv(kv, n_rep=4)  # -> 32 heads
print(f"KV original:  {kv.shape}  (8 heads)")
print(f"KV expanded:  {kv_expanded.shape}  (32 heads)")
print(f"\nMemory savings: KV cache stores 8 heads, not 32")
print(f"  MHA cache:  {32 * 64 * 4} bytes/token/layer")
print(f"  GQA cache:  {8 * 64 * 4} bytes/token/layer")
print(f"  Reduction:  {(1 - 8/32)*100:.0f}%")

In [ ]:
# Compare memory for different GQA ratios
n_q_heads = 32
head_dim = 128
seq_len = 4096
n_layers = 32
dtype_bytes = 2  # fp16

print(f"KV Cache Memory (Llama-like config, seq={seq_len}, {n_layers} layers, fp16):")
print(f"{'Config':<25} {'KV Heads':<10} {'Cache Size':<15} {'vs MHA':<10}")
print("-" * 60)

mha_bytes = 2 * n_layers * seq_len * n_q_heads * head_dim * dtype_bytes
for name, n_kv in [("MHA", 32), ("GQA-8", 8), ("GQA-4", 4), ("GQA-2", 2), ("MQA", 1)]:
    cache_bytes = 2 * n_layers * seq_len * n_kv * head_dim * dtype_bytes
    ratio = cache_bytes / mha_bytes
    print(f"{name:<25} {n_kv:<10} {cache_bytes/1024**2:<15.1f} MB  {ratio:.2f}x")

---
## 4. SwiGLU FFN

### Standard FFN
```
FFN(x) = ReLU(x @ W1) @ W2    (2 matrices)
```

### Modern FFN (SwiGLU)
```
FFN(x) = (SiLU(x @ W_gate) ⊙ (x @ W1)) @ W2    (3 matrices)
```

The gating mechanism lets the network learn which dimensions to pass through.

In [ ]:
class StandardFFN(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
    
    def forward(self, x):
        return self.w2(F.relu(self.w1(x)))


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.w_gate = nn.Linear(dim, hidden_dim, bias=False)
    
    def forward(self, x):
        return self.w2(F.silu(self.w_gate(x)) * self.w1(x))


dim = 512
hidden = 1365  # int(2/3 * 4 * dim) for SwiGLU

standard = StandardFFN(dim, 4 * dim)  # Standard uses 4x
swiglu = SwiGLU(dim, hidden)

std_params = sum(p.numel() for p in standard.parameters())
swi_params = sum(p.numel() for p in swiglu.parameters())

print(f"Standard FFN: {std_params:,} params  (2 matrices: {dim}x{4*dim} + {4*dim}x{dim})")
print(f"SwiGLU FFN:   {swi_params:,} params  (3 matrices: {dim}x{hidden} x3)")
print(f"\nSwiGLU uses 2/3 hidden dim to compensate for 3 matrices")
print(f"Result: roughly same param count but better training dynamics")

In [ ]:
# Visualize SiLU vs ReLU
x = torch.linspace(-4, 4, 200)
relu_y = F.relu(x)
silu_y = F.silu(x)

print("SiLU(x) = x * sigmoid(x)")
print("\nKey differences from ReLU:")
print(f"  SiLU(-1.0) = {F.silu(torch.tensor(-1.0)).item():.4f}  (ReLU = 0)")
print(f"  SiLU(0.0)  = {F.silu(torch.tensor(0.0)).item():.4f}  (ReLU = 0)")
print(f"  SiLU(1.0)  = {F.silu(torch.tensor(1.0)).item():.4f}  (ReLU = 1)")
print("\nSiLU is smooth everywhere (no dead neurons) and slightly negative for x < 0")

---
## 5. RMSNorm vs LayerNorm

RMSNorm is simpler and faster than LayerNorm:
- No mean computation (one less reduction)
- No bias parameter
- Just normalize by root-mean-square: `y = x / RMS(x) * γ`

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


dim = 4096
batch, seq = 4, 512
x = torch.randn(batch, seq, dim)

rms_norm = RMSNorm(dim)
layer_norm = nn.LayerNorm(dim)

# Timing comparison
n_iters = 100

start = time.perf_counter()
for _ in range(n_iters):
    _ = layer_norm(x)
ln_time = (time.perf_counter() - start) / n_iters

start = time.perf_counter()
for _ in range(n_iters):
    _ = rms_norm(x)
rms_time = (time.perf_counter() - start) / n_iters

print(f"LayerNorm: {ln_time*1000:.3f} ms")
print(f"RMSNorm:   {rms_time*1000:.3f} ms")
print(f"Speedup:   {ln_time/rms_time:.2f}x")
print(f"\nLayerNorm params: {sum(p.numel() for p in layer_norm.parameters())}")
print(f"RMSNorm params:   {sum(p.numel() for p in rms_norm.parameters())}")

---
## 6. Weight Tying

Share the embedding matrix with the output (language model head) matrix.
Both are `(vocab_size, dim)` — no reason to have two copies.

In [ ]:
vocab_size = 32000
dim = 4096

# Without tying
embedding = nn.Embedding(vocab_size, dim)
output_head = nn.Linear(dim, vocab_size, bias=False)
untied_params = embedding.weight.numel() + output_head.weight.numel()

# With tying
embedding_tied = nn.Embedding(vocab_size, dim)
output_tied = nn.Linear(dim, vocab_size, bias=False)
output_tied.weight = embedding_tied.weight  # Share!
tied_params = embedding_tied.weight.numel()  # Only one copy

print(f"Without weight tying: {untied_params:,} params ({untied_params * 2 / 1024**2:.0f} MB in fp16)")
print(f"With weight tying:    {tied_params:,} params ({tied_params * 2 / 1024**2:.0f} MB in fp16)")
print(f"Savings: {(untied_params - tied_params):,} params ({(untied_params - tied_params) * 2 / 1024**2:.0f} MB)")

# Verify they're the same tensor
assert output_tied.weight.data_ptr() == embedding_tied.weight.data_ptr()
print("\nVerified: same underlying memory!")

---
## 7. BFloat16 Training

bf16 has the same exponent range as fp32 (8 bits) but reduced mantissa (7 bits vs 23).
This means: no overflow issues, no loss scaler needed, simpler training code.

In [ ]:
# Compare fp16 vs bf16 ranges
print("Data Type Comparison:")
print(f"{'Type':<10} {'Max Value':<20} {'Min Normal':<20} {'Exponent bits':<15} {'Mantissa bits'}")
print("-" * 80)

fp16_max = torch.finfo(torch.float16).max
bf16_max = torch.finfo(torch.bfloat16).max
fp32_max = torch.finfo(torch.float32).max

print(f"{'fp16':<10} {fp16_max:<20.0f} {torch.finfo(torch.float16).tiny:<20.2e} {'5':<15} {'10'}")
print(f"{'bf16':<10} {bf16_max:<20.2e} {torch.finfo(torch.bfloat16).tiny:<20.2e} {'8':<15} {'7'}")
print(f"{'fp32':<10} {fp32_max:<20.2e} {torch.finfo(torch.float32).tiny:<20.2e} {'8':<15} {'23'}")

print(f"\nfp16 overflows at {fp16_max:.0f} — common in LLM gradients!")
print(f"bf16 max is {bf16_max:.2e} — same range as fp32, no scaler needed")

In [ ]:
# bf16 training pattern (simplified, works on CPU too)
model = nn.Linear(256, 256)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(4, 256)
target = torch.randn(4, 256)

# bf16 autocast — note: no GradScaler needed!
use_amp = torch.cuda.is_available()
amp_device = 'cuda' if use_amp else 'cpu'

with torch.amp.autocast(device_type=amp_device, dtype=torch.bfloat16, enabled=use_amp):
    out = model(x)
    loss = F.mse_loss(out, target)

loss.backward()
optimizer.step()
optimizer.zero_grad()

print("bf16 training step complete (no GradScaler needed!)")
print(f"Loss dtype: {loss.dtype}")
if use_amp:
    print("Used bf16 autocast on CUDA")
else:
    print("Running on CPU (autocast disabled, but pattern is the same)")

---
## 8. Gradient Accumulation

Simulate large batch sizes without needing the memory for a large batch.

In [ ]:
# Gradient accumulation demo
model = nn.Linear(64, 10)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

micro_batch_size = 4
accumulation_steps = 8
effective_batch_size = micro_batch_size * accumulation_steps

print(f"Micro-batch size: {micro_batch_size}")
print(f"Accumulation steps: {accumulation_steps}")
print(f"Effective batch size: {effective_batch_size}")
print()

# Simulate one optimizer step
optimizer.zero_grad()
for i in range(accumulation_steps):
    x = torch.randn(micro_batch_size, 64)
    y = torch.randint(0, 10, (micro_batch_size,))
    
    loss = F.cross_entropy(model(x), y) / accumulation_steps  # Normalize!
    loss.backward()  # Gradients accumulate in .grad
    
    if i == 0:
        grad_after_1 = model.weight.grad.clone()

# Now step
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

print(f"Grad norm after 1 micro-batch: {grad_after_1.norm():.4f}")
print(f"Grad norm after {accumulation_steps} micro-batches: {model.weight.grad.norm():.4f}")
print("\nGradients accumulated correctly (equivalent to one large batch)")

---
## 9. Sliding Window Attention

Only attend to the last W tokens. Reduces memory from O(n²) to O(n×W).

In [ ]:
def make_sliding_window_mask(seq_len: int, window_size: int) -> torch.Tensor:
    """Create sliding window causal attention mask."""
    row_idx = torch.arange(seq_len).unsqueeze(1)
    col_idx = torch.arange(seq_len).unsqueeze(0)
    valid = (col_idx <= row_idx) & (row_idx - col_idx < window_size)
    return torch.where(valid, 0.0, float('-inf'))


# Standard causal vs sliding window
seq_len = 12
window = 4

causal_mask = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)
window_mask = make_sliding_window_mask(seq_len, window)

print(f"Standard causal mask (seq={seq_len}):")
print("  Each token attends to ALL previous tokens")
for i in range(seq_len):
    row = ['.' if causal_mask[i,j] == float('-inf') else 'X' for j in range(seq_len)]
    print(f"  {i:2d}: {''.join(row)}")

print(f"\nSliding window mask (window={window}):")
print(f"  Each token attends to last {window} tokens only")
for i in range(seq_len):
    row = ['.' if window_mask[i,j] == float('-inf') else 'X' for j in range(seq_len)]
    print(f"  {i:2d}: {''.join(row)}")

causal_ops = sum((causal_mask[i] == 0).sum().item() for i in range(seq_len))
window_ops = sum((window_mask[i] == 0).sum().item() for i in range(seq_len))
print(f"\nAttention computations: causal={causal_ops}, window={window_ops} ({window_ops/causal_ops*100:.0f}%)")

---
## 10. Mini-LLM: Putting It All Together

Let's build a complete mini-LLM with ALL the techniques above.

In [ ]:
from dataclasses import dataclass

@dataclass
class MiniConfig:
    vocab_size: int = 256
    dim: int = 128
    n_layers: int = 2
    n_heads: int = 4
    n_kv_heads: int = 2  # GQA: 4 Q heads, 2 KV heads
    max_seq_len: int = 64
    rope_theta: float = 10000.0


class MiniGQA(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.n_rep = config.n_heads // config.n_kv_heads
        self.head_dim = config.dim // config.n_heads
        
        self.wq = nn.Linear(config.dim, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.dim, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.dim, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.dim, bias=False)
    
    def forward(self, x, freqs_cis, mask=None, cache=None):
        b, s, _ = x.shape
        q = self.wq(x).view(b, s, self.n_heads, self.head_dim)
        k = self.wk(x).view(b, s, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(b, s, self.n_kv_heads, self.head_dim)
        
        q, k = apply_rotary_emb(q, k, freqs_cis)
        
        if cache is not None:
            k, v = cache.update(k, v)
        
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)
        
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        scores = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores + mask
        out = torch.matmul(F.softmax(scores, dim=-1), v)
        return self.wo(out.transpose(1,2).reshape(b, s, -1))


class MiniBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = MiniGQA(config)
        self.ffn = SwiGLU(config.dim, int(2/3 * 4 * config.dim))
        self.norm1 = RMSNorm(config.dim)
        self.norm2 = RMSNorm(config.dim)
    
    def forward(self, x, freqs_cis, mask=None, cache=None):
        x = x + self.attn(self.norm1(x), freqs_cis, mask, cache)
        x = x + self.ffn(self.norm2(x))
        return x


class MiniLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.dim)
        self.layers = nn.ModuleList([MiniBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.dim)
        self.output = nn.Linear(config.dim, config.vocab_size, bias=False)
        self.output.weight = self.embedding.weight  # Weight tying!
        
        head_dim = config.dim // config.n_heads
        self.register_buffer('freqs_cis', 
            precompute_freqs_cis(head_dim, config.max_seq_len, config.rope_theta),
            persistent=False)
    
    def forward(self, tokens, caches=None):
        b, s = tokens.shape
        x = self.embedding(tokens)
        
        start_pos = caches[0].pos if (caches and caches[0].pos > 0) else 0
        freqs_cis = self.freqs_cis[start_pos:start_pos + s]
        
        mask = None
        if s > 1:
            mask = torch.triu(torch.full((s, s), float('-inf'), device=tokens.device), diagonal=1)
        
        for i, layer in enumerate(self.layers):
            cache = caches[i] if caches else None
            x = layer(x, freqs_cis, mask, cache)
        
        return self.output(self.norm(x))


config = MiniConfig()
model = MiniLLM(config)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Mini-LLM created!")
print(f"  Parameters: {n_params:,}")
print(f"  Techniques: RoPE + GQA + SwiGLU + RMSNorm + Weight Tying")

In [ ]:
# Train the mini-LLM
torch.manual_seed(42)
config = MiniConfig()
model = MiniLLM(config)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

accumulation_steps = 2
num_steps = 30

model.train()
print(f"Training for {num_steps} steps (grad accum = {accumulation_steps})...\n")

micro = 0
for step in range(num_steps * accumulation_steps):
    # Random data
    data = torch.randint(0, config.vocab_size, (2, config.max_seq_len + 1))
    x, y = data[:, :-1], data[:, 1:]
    
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, config.vocab_size), y.view(-1)) / accumulation_steps
    loss.backward()
    micro += 1
    
    if micro % accumulation_steps == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
        opt_step = micro // accumulation_steps
        if opt_step % 10 == 0:
            print(f"  Step {opt_step}/{num_steps} | Loss: {loss.item() * accumulation_steps:.4f}")

print("\nTraining complete!")

---
## 11. Generate Text with KV Cache

In [ ]:
@torch.no_grad()
def generate(model, prompt_tokens, max_new_tokens=20, temperature=0.8, top_k=50):
    """Generate tokens using KV cache."""
    config = model.config
    
    caches = [
        KVCache(1, config.max_seq_len, config.n_kv_heads, 
                config.dim // config.n_heads)
        for _ in range(config.n_layers)
    ]
    
    # Prefill
    logits = model(prompt_tokens, caches=caches)
    next_logits = logits[:, -1, :]
    
    generated = []
    for _ in range(max_new_tokens):
        scaled = next_logits / temperature
        if top_k > 0:
            topk_v, topk_i = scaled.topk(min(top_k, scaled.size(-1)))
            scaled = torch.full_like(scaled, float('-inf'))
            scaled.scatter_(1, topk_i, topk_v)
        
        probs = F.softmax(scaled, dim=-1)
        next_token = torch.multinomial(probs, 1)
        generated.append(next_token.item())
        
        next_logits = model(next_token, caches=caches)[:, -1, :]
    
    return generated


model.eval()
prompt = torch.randint(0, config.vocab_size, (1, 8))
print(f"Prompt tokens: {prompt[0].tolist()}")

start = time.perf_counter()
tokens = generate(model, prompt, max_new_tokens=20)
elapsed = time.perf_counter() - start

print(f"Generated tokens: {tokens}")
print(f"Time: {elapsed*1000:.1f} ms ({len(tokens)/elapsed:.0f} tokens/sec)")

In [ ]:
# Compare generation speed: cached vs uncached
prompt = torch.randint(0, config.vocab_size, (1, 16))
gen_tokens = 32

# Cached generation
start = time.perf_counter()
_ = generate(model, prompt, max_new_tokens=gen_tokens)
cached_time = time.perf_counter() - start

# Uncached: full forward at each step
start = time.perf_counter()
tokens_so_far = prompt.clone()
with torch.no_grad():
    for _ in range(gen_tokens):
        logits = model(tokens_so_far)
        probs = F.softmax(logits[:, -1, :] / 0.8, dim=-1)
        next_tok = torch.multinomial(probs, 1)
        tokens_so_far = torch.cat([tokens_so_far, next_tok], dim=1)
uncached_time = time.perf_counter() - start

print(f"Cached generation:   {cached_time*1000:.1f} ms")
print(f"Uncached generation: {uncached_time*1000:.1f} ms")
print(f"Speedup: {uncached_time/cached_time:.2f}x")

---
## 12. Exercise: Add Sliding Window Attention to the Mini-LLM

**Challenge**: Modify the `MiniGQA` class to support sliding window attention.

Hints:
1. Add a `window_size` parameter to the config
2. In `MiniGQA.forward`, replace the standard causal mask with a sliding window mask
3. For the KV cache during generation, you only need to keep the last `window_size` entries

Try implementing it below:

In [ ]:
# Exercise: Implement sliding window attention
# Uncomment and complete:

# @dataclass
# class SWAConfig(MiniConfig):
#     window_size: int = 16
#
# class SlidingWindowGQA(MiniGQA):
#     def __init__(self, config):
#         super().__init__(config)
#         self.window_size = config.window_size
#
#     def forward(self, x, freqs_cis, mask=None, cache=None):
#         # TODO: Create sliding window mask instead of full causal
#         # Hint: use make_sliding_window_mask(seq_len, self.window_size)
#         pass

print("Exercise: Implement sliding window attention in the mini-LLM")
print("See make_sliding_window_mask() above for the mask creation logic")

---
## Key Takeaways

| Technique | What It Does | Used In |
|-----------|-------------|--------|
| **RoPE** | Relative position via rotation | Llama, Mistral, GPT-NeoX |
| **KV Cache** | Cache K/V for O(n) generation | All autoregressive LLMs |
| **GQA** | Fewer KV heads → smaller cache | Llama 2 70B, Llama 3 |
| **Sliding Window** | Limit attention span | Mistral, Mixtral |
| **RMSNorm** | Faster normalization | Llama, Gemma |
| **SwiGLU** | Gated FFN with SiLU | Llama, PaLM |
| **Weight Tying** | Share embed/output | GPT-2, Llama |
| **bf16** | No loss scaler needed | All modern LLMs |
| **Grad Accumulation** | Large effective batch | All large-scale training |

### What's Next?
- **torchtune**: PyTorch's official library for LLM fine-tuning
- **FSDP2**: Scale to multi-GPU (Module 10)
- **torch.compile**: Optimize the model (Module 08)
- **FlexAttention**: Custom attention patterns (Module 09)

In [ ]:
print("Module 22 complete!")
print("\nYou've learned the building blocks of modern LLMs:")
print("  - RoPE for position encoding")
print("  - KV Cache for fast generation")
print("  - GQA for memory-efficient attention")
print("  - SwiGLU for better FFN")
print("  - RMSNorm for faster normalization")
print("  - Weight tying for parameter efficiency")
print("  - bf16 + gradient accumulation for training")
print("  - All combined in a working mini-LLM!")